<a href="https://colab.research.google.com/github/BrenoBragaGalvao/mestrado-arvores-urbanas/blob/main/Segmentacao_area_completa_v2_experimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentação de arvores e análise

## Preparação dos dados para segmentação:

Instalação das bibliotecas neessárias:

In [ ]:
!pip install rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Importação das funções e pacotes

In [ ]:
from skimage.segmentation import felzenszwalb, slic, quickshift, watershed
from skimage.segmentation import mark_boundaries
from skimage.util import img_as_float
from skimage.filters import sobel
from skimage.color import rgb2gray
from skimage import color
import rasterio
from matplotlib import pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
from rasterio.plot import show

Abrir a imagem com o rasterio, convertendo num numpy array:

In [ ]:
path_img = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_RGB_NEW.tif'
src_rgb = rasterio.open(path_img)
im = src_rgb.read()
im  = im.transpose([1,2,0])
'''R = src_rgb.read(3)
G = src_rgb.read(2)
B = src_rgb.read(1)


R = R / 10000
G = G / 10000
B = B / 10000

R = R * 255
G = G * 255
B = B * 255

R = R.astype('uint8')
G = G.astype('uint8')
B = B.astype('uint8')

im = np.dstack((R,G,B))'''

Plotando a imagem com o matplotlib:

In [ ]:
plt.figure(figsize=[12,12])
plt.imshow(im)
plt.axis('off')

Abrir o poligono da área de interesse:

In [ ]:
AOI = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_new.shp')

## Segmentação:

Aplicação do quickshift (Ver artigos relacionados)

In [ ]:
segments_quick2 = quickshift(im, kernel_size=3, max_dist=5, ratio=0.04)

In [ ]:
plt.figure(figsize=[16,16])
plt.imshow(mark_boundaries(im, segments_quick2))
plt.axis('off')

In [ ]:
segments_quick2 = segments_quick2.astype('uint16')

Conversão para poligono e extração das caracteristicas espectrais:

In [ ]:
from rasterio import features
from shapely.geometry import shape
import geopandas as gpd
import shapely
import json
from rasterio.mask import mask
import pandas as pd

Primeiro, converter os segmentos em poligonos usando geopandas e rasterio:

In [ ]:
find_shapes = features.shapes(segments_quick2, transform=src_rgb.transform)

In [ ]:
polygons = [shapely.geometry.Polygon(shape[0]["coordinates"][0]) for shape in find_shapes]

In [ ]:
crs = src_rgb.crs
Poly_gdf = gpd.GeoDataFrame(crs=crs, geometry=polygons)

Salvar os poligonos em um JSON:

In [ ]:
Poly_gdf.to_file("polygons.json")

In [ ]:
Poly_gdf.boundary.plot()

Obter somente os poligonos da área de interesse:

In [ ]:
clip_poly = Poly_gdf.clip(AOI)

In [ ]:
clip_poly.boundary.plot()

Extrair a média espectral de cada segmento:

In [ ]:
clip_poly = clip_poly.reset_index(drop=True)

In [ ]:
json_dataset = json.loads(clip_poly.to_json())['features']

In [ ]:
path_img_mult = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_NEW_MULTI.tif'
src = rasterio.open(path_img_mult)

X = []
Y = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=0)

  out_image = out_image.transpose([1,2,0])

  R_mean = out_image[:,:,0][np.nonzero(out_image[:,:,0])].mean()
  G_mean = out_image[:,:,1][np.nonzero(out_image[:,:,1])].mean()
  B_mean = out_image[:,:,2][np.nonzero(out_image[:,:,2])].mean()
  N_mean = out_image[:,:,3][np.nonzero(out_image[:,:,3])].mean()
  R_std = out_image[:,:,0][np.nonzero(out_image[:,:,0])].std()
  G_std = out_image[:,:,1][np.nonzero(out_image[:,:,1])].std()
  B_std = out_image[:,:,2][np.nonzero(out_image[:,:,2])].std()
  N_std = out_image[:,:,3][np.nonzero(out_image[:,:,3])].std()




  X.append([R_mean,G_mean,B_mean,N_mean,R_std,G_std,B_std,N_std])

In [ ]:
X = np.array(X)

## Aplicação do K-Means

Geramos o dataframe com a informação das médias espectrais por segmentos:

In [ ]:
df = pd.DataFrame(X, columns=['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std'])

In [ ]:
df

In [ ]:
df.dropna(inplace=True)

Aplicamos o Kmeans:

In [ ]:
from sklearn.cluster import KMeans

# Apply K-Means clustering
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10) # You can change the number of clusters
kmeans_clusters = kmeans.fit_predict(df[['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std']])

# Add K-Means cluster labels to the DataFrame
df['kmeans_cluster'] = kmeans_clusters

display(df.head())

Aplicamos o resultado do Kmeans para os segmentos e visualizamos:

In [ ]:
clip_poly['kmeans_cluster'] = df['kmeans_cluster']

In [ ]:
import matplotlib.pyplot as plt

# Plot the GeoDataFrame, colored by cluster
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
clip_poly.plot(column='kmeans_cluster', ax=ax, legend=True, cmap='magma') # Changed colormap to 'viridis'
plt.title('Polygons Colored by kmeans_cluster')
plt.axis('off')
plt.show()

Salvamos o resultado em um JSON:

In [ ]:
Poly_gdf.to_file('polygons.json')

In [ ]:
clip_poly[clip_poly['kmeans_cluster'] == 3].to_file('polygons_arvores.json')

Selecionamos somente o cluster de arvores:

In [ ]:
import rasterio.plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
clip_poly[clip_poly['kmeans_cluster'] == 3].boundary.plot(ax=ax, color='red')
plt.title('Original Image')
plt.axis('off')
plt.show()

In [ ]:
path_grama = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Grama.gpkg'
path_arvore = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Arvores.gpkg'

In [ ]:
gdf_grama = gpd.read_file(path_grama)
gdf_arvore = gpd.read_file(path_arvore)

In [ ]:
gdf_grama['Classe'] = 'grama'
gdf_arvore['Classe'] = 'arvore'

In [ ]:
gdf_full = pd.concat([gdf_grama, gdf_arvore])
gdf_full.to_crs(clip_poly.crs, inplace=True)

In [ ]:
gdf_arvore_seg = clip_poly[clip_poly['kmeans_cluster'] == 3]

In [ ]:
gdf_arvore_seg_filter = gdf_arvore_seg.sjoin(gdf_full, how='inner', predicate='intersects')

In [ ]:
gdf_arvore_seg_filter.plot()

In [ ]:
gdf_arvore_seg_filter

In [ ]:
src = rasterio.open(path_img_mult)

In [ ]:
img = src.read()

In [ ]:
img.shape

In [ ]:
nir = img[3]
red = img[0]

In [ ]:
nir.shape

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(nir, cmap='gray')
plt.title('NIR Band')
plt.axis('off')
plt.show()

In [ ]:
import numpy as np

# Calculate NDVI
# Avoid division by zero by adding a small epsilon to the denominator
ndvi = (nir.astype(float) - red.astype(float)) / (nir.astype(float) + red.astype(float) + 1e-8)

# Plot NDVI
plt.figure(figsize=(10, 10))
plt.imshow(ndvi, cmap='RdYlGn', vmin=0, vmax=1) # RdYlGn is a common colormap for NDVI
plt.colorbar(label='NDVI Value')
plt.title('Normalized Difference Vegetation Index (NDVI)')
plt.axis('off')
plt.show()

In [ ]:
nir.max()

In [ ]:
from skimage.feature import canny
from skimage.util import img_as_ubyte

# Ensure nir is in the correct format (uint8) for Canny
nir_uint = nir/10000

# Apply Canny edge detection
edges = canny(nir_uint, sigma=1)

# Plot the edges
plt.figure(figsize=(10, 10))
plt.imshow(edges, cmap='gray')
plt.title('Canny Edges of NIR Band')
plt.axis('off')
plt.show()

In [ ]:
img_rgb = img[0:3]

In [ ]:
img_rgb = img_rgb.transpose([1,2,0])

In [ ]:
from skimage.color import rgb2hsv

# Convert RGB to HSV
img_hsv = rgb2hsv(img_rgb)

# Extract the intensity (Value) channel, which is the third channel (index 2)
intensity = img_hsv[:, :, 2]

# Display the intensity image
plt.figure(figsize=(10, 10))
plt.imshow(intensity, cmap='gray')
plt.title('Intensity Channel (HSV)')
plt.axis('off')
plt.show()

In [ ]:
# Convert edges (boolean) to float for stacking
edges_float = edges.astype(np.float32)

# Stack intensity and edges as two separate bands
# The order here is intensity first, then edges
stacked_bands = np.stack([intensity, edges_float, ndvi], axis=0)

# Get metadata from the original source (src) for writing the new TIFF
profile = src.profile

# Update profile for the new stacked image
# Change the number of bands to 2 and data type to float32
profile.update(
    dtype=rasterio.float32,
    count=3,
    nodata=None
)

# Define the output path for the new TIFF file
output_filepath = 'features_stack.tif'

# Write the stacked image to a new TIFF file
with rasterio.open(output_filepath, 'w', **profile) as dst:
    dst.write(stacked_bands)


In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg_filter.to_json())['features']

X = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)
  classe_values = json_dataset[i]['properties']['Classe']

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X.append([mean_intensity, mean_edges, mean_ndvi,classe_values])


In [ ]:
df = pd.DataFrame(X, columns=['mean_intensity','mean_edges','mean_ndvi','classe'])

In [ ]:
import seaborn as sns

# Create a pairplot with 'classe' as hue
sns.pairplot(df, hue='classe')
plt.show()

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Separate features (X) and target (y)
X = df.drop('classe', axis=1) # Features are all columns except 'classe'
y = df['classe'] # Target is the 'classe' column

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Classifier
# You can adjust parameters like n_estimators (number of trees) as needed
rf_classifier = RandomForestClassifier(n_estimators=500, random_state=42)

# Train the model
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test)

# Evaluate the model
print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}")

In [ ]:
# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=rf_classifier.classes_)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_classifier.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg.to_json())['features']

X_pred = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X_pred.append([mean_intensity, mean_edges, mean_ndvi])

In [ ]:
df_pred = pd.DataFrame(X_pred, columns=['mean_intensity','mean_edges','mean_ndvi'])

In [ ]:
gdf_arvore_seg['predicted_classe'] = rf_classifier.predict(df_pred)

In [ ]:
gdf_arvore_seg

In [ ]:
gdf_arvore_seg['area'] = gdf_arvore_seg.area

In [ ]:
tree_polys = gdf_arvore_seg.copy()

## Seleção das arvores e aplicação da Alometria:

Selecionamos os segmentos que pertence ao cluster das arvores:

In [ ]:
tree_polys

Aplicamos a formula da alometria:

In [ ]:
import numpy as np

#C1
a1 = 0.5
b1 = 0.7
tree_polys['Power_tree_size_C1'] = a1 * (tree_polys['area'] ** b1)
tree_polys['Linear_tree_size_C1'] = a1 + (tree_polys['area'] * b1)


#C2
a2 = 0.6
b2 = 0.8
tree_polys['Power_tree_size_C2'] = a2 * (tree_polys['area'] ** b2)
tree_polys['Linear_tree_size_C2'] = a2 + (tree_polys['area'] * b2)


#C3
a3 = 2.5
b3 = 0.4
tree_polys['Power_tree_size_C3'] = a3 * (tree_polys['area'] ** b3)
tree_polys['Linear_tree_size_C3'] = a3 + (tree_polys['area'] * b3)

#C4
a4 = 4
b4 = 0.5
tree_polys['Power_tree_size_C4'] = a4 * (tree_polys['area'] ** b4)
tree_polys['Linear_tree_size_C4'] = a4 + (tree_polys['area'] * b4)

#C5

a5 = 0.4
b5 = 0.9
tree_polys['Power_tree_size_C5'] = a5 * (tree_polys['area'] ** b5)
tree_polys['Linear_tree_size_C5'] = a5 + (tree_polys['area'] * b5)



a6 = 1.35
b6 = 0.72
tree_polys['Power_tree_size_C6'] = a6 * (tree_polys['area'] ** b6)
tree_polys['Linear_tree_size_C6'] = a6 + (tree_polys['area'] * b6)

In [ ]:
columns = ['Power_tree_size_C1', 'Linear_tree_size_C1', 'Power_tree_size_C2', 'Linear_tree_size_C2',
           'Power_tree_size_C3', 'Linear_tree_size_C3', 'Power_tree_size_C4', 'Linear_tree_size_C4',
           'Power_tree_size_C5', 'Linear_tree_size_C5', 'Power_tree_size_C6', 'Linear_tree_size_C6']

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys[columns])
plt.title('Box Plot of Tree Size Metrics for Different Constants')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys['Power_tree_size_C3'])
plt.title('Box Plot of Tree Size Metrics for C3')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(tree_polys['Power_tree_size_C3'], bins=30, edgecolor='black')
plt.title('Histogram of Power_tree_size_C3')
plt.xlabel('Power_tree_size_C3')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

Obtemos somente arvores maiores que 7 metros:

In [ ]:
filter_tree = tree_polys[tree_polys['Power_tree_size_C3'] > 7]

Salvamos em um arquivo JSON:

In [ ]:
filter_tree.to_file('arvores_segmentadas.json')

## Análise das arvores proximas a linhas e postes:

Importação e visualização das linhas e postes:

In [ ]:
Postes = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/postes.gpkg')

Arvores maiores que 7m a 1m de distancia ou menos de postes:

In [ ]:
Postes = Postes.to_crs(filter_tree.crs)

In [ ]:
Postes = Postes.clip(AOI)

### Arvores e postes a 1m

In [ ]:
postes_prox_tree_1m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=1)

In [ ]:
arvores_prox_postes_1m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_1m.boundary.plot(ax=ax, color='green')
postes_prox_tree_1m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

### Arvores e postes a 3m

In [ ]:
postes_prox_tree_3m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=3)

In [ ]:
arvores_prox_postes_3m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=3)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_3m.boundary.plot(ax=ax, color='green')
postes_prox_tree_3m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

### Arvores e postes a 5m

In [ ]:
postes_prox_tree_5m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=5)

In [ ]:
arvores_prox_postes_5m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=5)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_5m.boundary.plot(ax=ax, color='green')
postes_prox_tree_5m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

## V2 — Auditoria de amostras e avaliação sem duplicação de segmentos

**Etapa 2.** As células anteriores preservam integralmente o código original; apenas as saídas antigas foram limpas nesta cópia experimental (continuam no notebook original). Execute as células anteriores para reconstruir `df` e `gdf_arvore_seg_filter`, depois execute as células a seguir.

A auditoria procura (1) segmentos repetidos pelo `sjoin(intersects)`, (2) rótulos conflitantes e (3) sobreposição de identificadores de segmentos entre treino e teste original. O novo teste é exploratório, usa somente `Area1`, exclui explicitamente os segmentos com rótulos conflitantes, mantém uma linha por segmento e executa cinco separações por grupo. Não permite inferir risco elétrico, nem resolve autocorrelação espacial entre segmentos vizinhos. Nenhum resultado novo foi executado ou validado ao publicar este notebook.


In [ ]:
# V2 — auditoria da independência das amostras (não modifica o RF original)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

atributos_v2 = ["mean_intensity", "mean_edges", "mean_ndvi"]
assert all(c in df.columns for c in atributos_v2 + ["classe"])
assert len(df) == len(gdf_arvore_seg_filter), "Linhas desalinhadas entre df e GeoDataFrame."
assert np.array_equal(df["classe"].astype(str).to_numpy(),
                      gdf_arvore_seg_filter["Classe"].astype(str).to_numpy()), (
    "Classes ou ordem das linhas não coincidem; interrompa a auditoria."
)

# O spatial join preserva o índice do segmento da tabela à esquerda.
segmentos_v2 = pd.Series(gdf_arvore_seg_filter.index.to_numpy(), name="segmento")
pares_v2 = pd.DataFrame({
    "segmento": segmentos_v2,
    "classe": df["classe"].astype(str).reset_index(drop=True)
})
conflitos_v2 = pares_v2.groupby("segmento")["classe"].nunique()
ids_conflitantes_v2 = conflitos_v2[conflitos_v2 > 1].index

indices_tr_v2, indices_te_v2 = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42
)
ids_em_ambos_v2 = set(segmentos_v2.iloc[indices_tr_v2]) & set(segmentos_v2.iloc[indices_te_v2])
print("=== Diagnóstico da divisão original ===")
print("Pares objeto-rótulo:", len(df))
print("Segmentos únicos:", pares_v2["segmento"].nunique())
print("Registros repetidos por segmento:", len(df) - pares_v2["segmento"].nunique())
print("Segmentos com classes conflitantes:", len(ids_conflitantes_v2))
print("Segmentos compartilhados no treino e teste originais:", len(ids_em_ambos_v2))
print("Registros de teste desses segmentos:", int(
    segmentos_v2.iloc[indices_te_v2].isin(ids_em_ambos_v2).sum()))
print("Nota: segmentos vizinhos podem continuar espacialmente correlacionados.")


In [ ]:
# V2 — teste exploratório: 1 linha por segmento, cinco divisões agrupadas
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

pares_unicos_v2 = pares_v2.loc[
    ~pares_v2["segmento"].isin(ids_conflitantes_v2)
].drop_duplicates("segmento")
print("Registros excluídos (repetições/conflitos):", len(df) - len(pares_unicos_v2))
X_agr_v2 = df.loc[pares_unicos_v2.index, atributos_v2].to_numpy(dtype=float)
y_agr_v2 = pares_unicos_v2["classe"].to_numpy()
g_agr_v2 = pares_unicos_v2["segmento"].to_numpy()
if len(pares_unicos_v2) < 10 or len(np.unique(y_agr_v2)) < 2:
    raise ValueError("Não há segmentos distintos suficientes com ambas as classes.")
if not np.isfinite(X_agr_v2).all():
    raise ValueError("Atributos não finitos; reveja as extrações antes da avaliação.")

linhas_v2 = []
cv_v2 = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
for rodada, (tr, te) in enumerate(cv_v2.split(X_agr_v2, y_agr_v2, groups=g_agr_v2), start=1):
    if len(np.unique(y_agr_v2[tr])) < 2 or len(np.unique(y_agr_v2[te])) < 2:
        print(f"Rodada {rodada} ignorada: falta classe no treino ou no teste.")
        continue
    modelo_auditoria_v2 = RandomForestClassifier(
        n_estimators=500, random_state=42, n_jobs=-1)
    modelo_auditoria_v2.fit(X_agr_v2[tr], y_agr_v2[tr])
    pred_v2 = modelo_auditoria_v2.predict(X_agr_v2[te])
    linhas_v2.append({
        "rodada": rodada,
        "n_treino": len(tr),
        "n_teste": len(te),
        "acuracia": accuracy_score(y_agr_v2[te], pred_v2),
        "acuracia_balanceada": balanced_accuracy_score(y_agr_v2[te], pred_v2),
        "f1_macro": f1_score(y_agr_v2[te], pred_v2, average="macro", zero_division=0)
    })
resultado_auditoria_v2 = pd.DataFrame(linhas_v2)
if resultado_auditoria_v2.empty:
    print("Não houve nenhuma rodada válida; métricas não disponíveis.")
else:
    print(resultado_auditoria_v2.to_string(index=False))
    print("\nMédia e desvio-padrão (rodadas válidas):")
    print(resultado_auditoria_v2[["acuracia", "acuracia_balanceada", "f1_macro"]]
          .agg(["mean", "std"]).to_string())
    print("\nATENÇÃO: não comparar diretamente com 0,87, pois os testes diferem.")


## E2 — validação espacial por blocos com a mesma Area1

**Etapa seguinte à auditoria E1.** Treino e teste agora são separados em *blocos geográficos*, não apenas por registros. Uma faixa de exclusão de **25 metros** ao redor dos polígonos de teste evita que amostras de treinamento fiquem imediatamente encostadas nos objetos avaliados.

Usa somente `df` e `gdf_arvore_seg_filter` produzidos acima (imagens/rótulos existentes na `Area1`). Remove segmentos repetidos e aqueles com rótulos conflitantes, como no E1, preservando a classificação original e seus resultados históricos. O RF mantém **500 árvores**, `random_state=42` e os três atributos originais.

Testa blocos de **250 m** e **500 m**, com até 10 partições por escala. Cada partição contém blocos inteiros no teste; se não houver duas classes ou dados suficientes, a partição é descartada e isso é relatado. A proporção final de objetos em teste pode diferir de 20%, pois o sorteio é por blocos.

**Atenção:** separações dentro da mesma área de interesse são validação espacial interna, não teste de generalização para outra cidade. As métricas só passam a existir após executar as células com os dados reais; não são substitutas das métricas históricas de 0,87.


In [ ]:
# E2a — preparar dados com objetos únicos e geometria espacial real (Area1)
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score
)

campos_e2 = ["mean_intensity", "mean_edges", "mean_ndvi"]
assert len(df) == len(gdf_arvore_seg_filter), "As linhas de df e do sjoin não coincidem."
assert all(x in df.columns for x in campos_e2 + ["classe"])
assert "Classe" in gdf_arvore_seg_filter.columns
assert gdf_arvore_seg_filter.crs is not None, "É necessário um CRS conhecido."
rotulos_df_e2 = df["classe"].astype(str).str.strip().str.lower().to_numpy()
rotulos_geo_e2 = gdf_arvore_seg_filter["Classe"].astype(str).str.strip().str.lower().to_numpy()
assert np.array_equal(rotulos_df_e2, rotulos_geo_e2), (
    "Ordem ou classes diferem entre df e gdf; auditoria interrompida."
)

dados_e2 = df[campos_e2].reset_index(drop=True).copy()
dados_e2["classe"] = rotulos_df_e2
# O índice à esquerda do sjoin é o identificador do segmento; pode se repetir.
dados_e2["segmento_id"] = gdf_arvore_seg_filter.index.astype(str).to_numpy()
dados_e2["geometry"] = list(gdf_arvore_seg_filter.geometry.to_numpy())
base_e2 = gpd.GeoDataFrame(
    dados_e2, geometry="geometry", crs=gdf_arvore_seg_filter.crs
)
n_classes_e2 = base_e2.groupby("segmento_id")["classe"].nunique()
ids_conflitantes_e2 = set(n_classes_e2[n_classes_e2 > 1].index)
qtd_pares_e2 = len(base_e2)
base_e2 = (base_e2.loc[~base_e2["segmento_id"].isin(ids_conflitantes_e2)]
           .drop_duplicates(subset=["segmento_id"]).copy())
for campo_e2 in campos_e2:
    base_e2[campo_e2] = pd.to_numeric(base_e2[campo_e2], errors="coerce")
base_e2[campos_e2] = base_e2[campos_e2].replace([np.inf, -np.inf], np.nan)
base_e2 = base_e2.dropna(subset=campos_e2 + ["geometry"]).reset_index(drop=True)
if base_e2.geometry.is_empty.any() or (~base_e2.geometry.is_valid).any():
    raise ValueError("Polígonos vazios ou inválidos; revisar geometrias antes da avaliação.")
# EPSG:31982 é o CRS métrico dos rasters originais da pesquisa.
base_e2 = base_e2.to_crs(epsg=31982)
if len(base_e2) < 20 or base_e2["classe"].nunique() != 2:
    raise ValueError("Dados insuficientes para avaliar as duas classes por blocos.")
print("=== E2 | Preparação para validação espacial ===")
print("Pares objeto-rótulo anteriores:", qtd_pares_e2)
print("Segmentos conflitantes excluídos:", len(ids_conflitantes_e2))
print("Objetos distintos e válidos:", len(base_e2))
print("Classes:", base_e2["classe"].value_counts().to_dict())
print("CRS métrico:", base_e2.crs)


In [ ]:
# E2b — separação espacial por blocos, embargo geométrico de 25 m e RF original
# Não baixa novos dados nem altera o modelo ou as variáveis do experimento histórico.
resultados_e2 = []
ignorados_e2 = []
tam_blocos_e2 = [250, 500]  # metros
faixa_exclusao_e2 = 25  # metros
centros_e2 = base_e2.geometry.centroid
xmin_e2, ymin_e2, _, _ = base_e2.total_bounds
X_e2 = base_e2[campos_e2].to_numpy(dtype=float)
y_e2 = base_e2["classe"].to_numpy()
classe_arvore_e2 = "arvore" if "arvore" in set(y_e2) else None

for tamanho_e2 in tam_blocos_e2:
    col_e2 = np.floor((centros_e2.x.to_numpy() - xmin_e2) / tamanho_e2).astype(int)
    lin_e2 = np.floor((centros_e2.y.to_numpy() - ymin_e2) / tamanho_e2).astype(int)
    grupos_e2 = np.array([f"{x}_{y}" for x, y in zip(col_e2, lin_e2)])
    total_blocos_e2 = len(np.unique(grupos_e2))
    if total_blocos_e2 < 5:
        ignorados_e2.append((tamanho_e2, 0, "menos de cinco blocos distintos"))
        continue

    splitter_e2 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e2, (tr_e2, te_e2) in enumerate(
        splitter_e2.split(X_e2, y_e2, groups=grupos_e2), start=1
    ):
        assert not (set(grupos_e2[tr_e2]) & set(grupos_e2[te_e2]))
        assert not (
            set(base_e2.iloc[tr_e2]["segmento_id"])
            & set(base_e2.iloc[te_e2]["segmento_id"])
        )
        # Excluir do treino objetos a até 25m das geometrias do teste.
        faixa_e2 = base_e2.geometry.iloc[te_e2].buffer(faixa_exclusao_e2)
        area_teste_e2 = (faixa_e2.union_all() if hasattr(faixa_e2, "union_all")
                         else faixa_e2.unary_union)
        manter_e2 = ~base_e2.geometry.iloc[tr_e2].intersects(area_teste_e2).to_numpy()
        tr_seguro_e2 = tr_e2[manter_e2]
        if len(tr_seguro_e2) < 15 or len(te_e2) < 5:
            ignorados_e2.append((tamanho_e2, rodada_e2, "amostra insuficiente após exclusão"))
            continue
        if (len(np.unique(y_e2[tr_seguro_e2])) < 2
                or len(np.unique(y_e2[te_e2])) < 2):
            ignorados_e2.append((tamanho_e2, rodada_e2, "uma classe ausente em treino ou teste"))
            continue

        rf_e2 = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
        rf_e2.fit(X_e2[tr_seguro_e2], y_e2[tr_seguro_e2])
        previsto_e2 = rf_e2.predict(X_e2[te_e2])
        linha_e2 = {
            "bloco_m": tamanho_e2, "rodada": rodada_e2,
            "blocos_total": total_blocos_e2,
            "blocos_teste": len(np.unique(grupos_e2[te_e2])),
            "n_treino": len(tr_seguro_e2), "n_teste": len(te_e2),
            "n_excluidos_faixa": len(tr_e2) - len(tr_seguro_e2),
            "teste_arvore": int((y_e2[te_e2] == "arvore").sum()),
            "teste_grama": int((y_e2[te_e2] == "grama").sum()),
            "acuracia": accuracy_score(y_e2[te_e2], previsto_e2),
            "acuracia_balanceada": balanced_accuracy_score(y_e2[te_e2], previsto_e2),
            "f1_macro": f1_score(y_e2[te_e2], previsto_e2, average="macro", zero_division=0),
        }
        if classe_arvore_e2 is not None:
            linha_e2["precisao_arvore"] = precision_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
            linha_e2["recall_arvore"] = recall_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
            linha_e2["f1_arvore"] = f1_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
        resultados_e2.append(linha_e2)

tabela_e2 = pd.DataFrame(resultados_e2)
print("=== E2 | Resultados por separação geográfica ===")
if tabela_e2.empty:
    print("Nenhuma rodada foi viável com os blocos, as classes e a faixa definidos.")
else:
    print(tabela_e2.round(3).to_string(index=False))
    print("\nResumo descritivo por tamanho de bloco:")
    print(tabela_e2.groupby("bloco_m")[["acuracia", "acuracia_balanceada", "f1_macro"]]
          .agg(["count", "mean", "std"]).round(3).to_string())
print("\nRodadas ignoradas:", len(ignorados_e2))
for tamanho_e2, rodada_e2, motivo_e2 in ignorados_e2:
    print(f"  Bloco {tamanho_e2}m; rodada {rodada_e2}: {motivo_e2}")
print("\nINTERPRETAÇÃO: repetição de divisões na mesma área NÃO equivale a")
print("testes em novas cidades; desvios-padrão entre rodadas não são intervalos")
print("de confiança independentes nem comprovam ganho sobre a acurácia 0,87 histórica.")


## E3 — comparação pareada de classificadores (somente Area1)

**Objetivo:** comparar os resultados de **Random Forest (RF)**, **SVM de kernel RBF**, **Regressão Logística** e uma referência ingênua que sempre prevê a classe majoritária (**DummyClassifier**), utilizando **exatamente os mesmos registros de teste em cada rodada**.

- Reutiliza os segmentos únicos e as três características `mean_intensity` (brilho HSV), `mean_edges` (bordas Canny) e `mean_ndvi` (NDVI), preparados em **E2a**.
- Reproduz o protocolo do **E2b**: blocos de 250 e 500 m, 10 tentativas por escala, semente 42, 20% dos **blocos** reservados para teste e exclusão de objetos de treino a até 25 m da geometria de teste.
- SVM e regressão logística aplicam `StandardScaler` **ajustado exclusivamente no treino**; o RF usa os mesmos 500 estimadores do experimento original.
- Não há busca de hiperparâmetros no conjunto de teste. Os quatro métodos utilizam as mesmas entradas e divisão; são classificadores de **árvore versus gramínea**, não detectores completos comparáveis a YOLO sem anotações adicionais.
- Repetições de teste compartilham amostras; as diferenças pareadas são descritivas, **não** testes independentes de significância.
- A análise continua restrita à **mesma área Area1**; os dados atuais não permitem alegar generalização externa, validar altura real ou interferência nos cabos.

**Execução:** rode as células originais e **E2a**, depois as duas células E3 a seguir. A execução de **E2b** não é obrigatória, mas, se tiver sido concluída, o E3 também confere a consistência de suas métricas do RF. Nenhuma métrica nova é presumida antes de executar o Colab.


In [ ]:
# E3a — RF vs SVM vs regressão logística vs baseline, nas MESMAS partições espaciais de E2
# Pré-requisito: E2a concluída; não altera rf_classifier, df ou os dados originais.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
)
import numpy as np
import pandas as pd

if "base_e2" not in globals() or "campos_e2" not in globals():
    raise RuntimeError("Execute E2a para criar base_e2/campos_e2 antes de E3.")
assert base_e2.crs is not None and base_e2.crs.is_projected, (
    "A geometria de E3 deve usar um CRS projetado com unidades em metros."
)
assert len(base_e2) == base_e2["segmento_id"].nunique(), (
    "Há segmentos repetidos; verifique a preparação E2a."
)
assert set(base_e2["classe"].unique()) == {"arvore", "grama"}, (
    "Esperadas as classes 'arvore' e 'grama' em E3."
)

X_e3 = base_e2[campos_e2].to_numpy(dtype=float)
y_e3 = base_e2["classe"].to_numpy()
if not np.isfinite(X_e3).all():
    raise ValueError("Atributos não finitos em E3.")
centros_e3 = base_e2.geometry.centroid
xmin_e3, ymin_e3, _, _ = base_e2.total_bounds
resultados_e3, ignorados_e3 = [], []
protocolos_validos_e3 = []

for tamanho_e3 in [250, 500]:
    col_e3 = np.floor((centros_e3.x.to_numpy() - xmin_e3) / tamanho_e3).astype(int)
    lin_e3 = np.floor((centros_e3.y.to_numpy() - ymin_e3) / tamanho_e3).astype(int)
    grupos_e3 = np.array([f"{c}_{l}" for c, l in zip(col_e3, lin_e3)])
    n_blocos_e3 = len(np.unique(grupos_e3))
    if n_blocos_e3 < 5:
        ignorados_e3.append((tamanho_e3, 0, "menos de cinco blocos"))
        continue

    separador_e3 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e3, (treino_e3, teste_e3) in enumerate(
        separador_e3.split(X_e3, y_e3, groups=grupos_e3), start=1
    ):
        if set(grupos_e3[treino_e3]) & set(grupos_e3[teste_e3]):
            raise AssertionError("Vazamento de blocos geográficos entre treino e teste.")
        if (set(base_e2.iloc[treino_e3]["segmento_id"]) &
                set(base_e2.iloc[teste_e3]["segmento_id"])):
            raise AssertionError("Mesmo segmento em treino e teste.")
        # Mesmo embargo espacial aplicado em E2b.
        buffers_e3 = base_e2.geometry.iloc[teste_e3].buffer(25)
        zona_e3 = (buffers_e3.union_all() if hasattr(buffers_e3, "union_all")
                   else buffers_e3.unary_union)
        afastados_e3 = ~base_e2.geometry.iloc[treino_e3].intersects(zona_e3).to_numpy()
        treino_final_e3 = treino_e3[afastados_e3]

        if len(treino_final_e3) < 15 or len(teste_e3) < 5:
            ignorados_e3.append((tamanho_e3, rodada_e3, "amostra insuficiente"))
            continue
        if (len(np.unique(y_e3[treino_final_e3])) != 2
                or len(np.unique(y_e3[teste_e3])) != 2):
            ignorados_e3.append((tamanho_e3, rodada_e3, "faltou uma classe"))
            continue

        modelos_e3 = {
            "Dummy (maioria)": DummyClassifier(strategy="most_frequent"),
            "Regressão logística": make_pipeline(
                StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)
            ),
            "SVM (RBF)": make_pipeline(
                StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=500, random_state=42, n_jobs=-1
            ),
        }
        protocolos_validos_e3.append((tamanho_e3, rodada_e3))
        for nome_e3, modelo_e3 in modelos_e3.items():
            modelo_e3.fit(X_e3[treino_final_e3], y_e3[treino_final_e3])
            previsoes_e3 = modelo_e3.predict(X_e3[teste_e3])
            resultados_e3.append({
                "bloco_m": tamanho_e3, "rodada": rodada_e3, "modelo": nome_e3,
                "n_treino": len(treino_final_e3), "n_teste": len(teste_e3),
                "n_excluidos_faixa": len(treino_e3) - len(treino_final_e3),
                "teste_arvore": int(np.sum(y_e3[teste_e3] == "arvore")),
                "teste_grama": int(np.sum(y_e3[teste_e3] == "grama")),
                "acuracia": accuracy_score(y_e3[teste_e3], previsoes_e3),
                "acuracia_balanceada": balanced_accuracy_score(
                    y_e3[teste_e3], previsoes_e3
                ),
                "f1_macro": f1_score(
                    y_e3[teste_e3], previsoes_e3, average="macro", zero_division=0
                ),
                "precisao_arvore": precision_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
                "recall_arvore": recall_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
                "f1_arvore": f1_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
            })

tabela_e3 = pd.DataFrame(resultados_e3)
print("=== E3 | Protocolo pareado entre classificadores ===")
print("Partições espaciais válidas:", len(protocolos_validos_e3))
print("Partições ignoradas:", len(ignorados_e3))
for b_e3, r_e3, motivo_e3 in ignorados_e3:
    print(f"  {b_e3} m | rodada {r_e3}: {motivo_e3}")

if tabela_e3.empty:
    print("Nenhuma partição permitiu comparar os modelos; não há métricas E3.")
else:
    n_modelos_e3 = tabela_e3.groupby(["bloco_m", "rodada"])["modelo"].nunique()
    assert (n_modelos_e3 == 4).all(), "Algum modelo não participou da mesma partição."
    print("\nMétricas por partição e algoritmo:")
    print(tabela_e3.round(3).to_string(index=False))
    print("\nResumo por escala e algoritmo — média e desvio-padrão descritivos:")
    resumo_e3 = tabela_e3.groupby(["bloco_m", "modelo"])[
        ["acuracia", "acuracia_balanceada", "f1_macro", "f1_arvore"]
    ].agg(["count", "mean", "std"]).round(3)
    print(resumo_e3.to_string())

    # Se E2b foi executado, suas métricas do RF devem coincidir na mesma partição.
    if ("tabela_e2" in globals() and isinstance(tabela_e2, pd.DataFrame)
            and not tabela_e2.empty):
        rf_e3 = tabela_e3.loc[tabela_e3["modelo"] == "Random Forest"]
        verif_e3 = rf_e3.merge(
            tabela_e2[["bloco_m", "rodada", "n_treino", "n_teste", "f1_macro"]],
            on=["bloco_m", "rodada"], how="outer",
            suffixes=("_e3", "_e2"), indicator=True
        )
        if not (verif_e3["_merge"] == "both").all():
            raise AssertionError("E2 e E3 não usaram as mesmas partições válidas.")
        assert np.array_equal(verif_e3["n_treino_e3"], verif_e3["n_treino_e2"])
        assert np.array_equal(verif_e3["n_teste_e3"], verif_e3["n_teste_e2"])
        assert np.allclose(verif_e3["f1_macro_e3"], verif_e3["f1_macro_e2"])
        print("\nConferência E2 × E3: mesmas divisões e métricas RF confirmadas.")

print("\nIMPORTANTE: não comparar diretamente estes valores com a acurácia 0,87")
print("histórica, calculada sob protocolo diferente.")


In [ ]:
# E3b — diferenças PAREADAS (mesmo teste por rodada); tabela e gráfico exploratório
# Não interpreta rodadas repetidas como observações estatisticamente independentes.
import matplotlib.pyplot as plt

if tabela_e3.empty:
    print("Sem métricas E3 para resumir ou representar graficamente.")
else:
    f1_pares_e3 = tabela_e3.pivot(
        index=["bloco_m", "rodada"], columns="modelo", values="f1_macro"
    )
    bal_pares_e3 = tabela_e3.pivot(
        index=["bloco_m", "rodada"], columns="modelo",
        values="acuracia_balanceada"
    )
    for alternativa_e3 in ["SVM (RBF)", "Regressão logística", "Dummy (maioria)"]:
        diferenca_e3 = f1_pares_e3["Random Forest"] - f1_pares_e3[alternativa_e3]
        print(f"\nDiferença de F1-macro por rodada: RF − {alternativa_e3}")
        print(diferenca_e3.groupby(level="bloco_m").agg(["count", "mean", "std"])
              .round(3).to_string())
    print("\nAcurácia balanceada: RF e concorrentes (mesmos testes):")
    print(bal_pares_e3.groupby(level="bloco_m").mean().round(3).to_string())

    for bloco_graf_e3, recorte_e3 in tabela_e3.groupby("bloco_m"):
        ordem_e3 = ["Dummy (maioria)", "Regressão logística", "SVM (RBF)",
                     "Random Forest"]
        media_e3 = recorte_e3.groupby("modelo")["f1_macro"].mean().reindex(ordem_e3)
        plt.figure(figsize=(9, 4))
        plt.bar(media_e3.index, media_e3.to_numpy())
        plt.ylim(0, 1)
        plt.ylabel("F1-macro médio nas partições válidas")
        plt.title(f"E3 — classificação com blocos espaciais de {bloco_graf_e3} m")
        plt.xticks(rotation=18, ha="right")
        plt.tight_layout()
        plt.show()

    print("\nConclusões dependem dos valores reais calculados após executar no Colab.")
    print("A comparação mede classificação árvore/grama na Area1 e não risco elétrico.")


## E4 — ablação de atributos: NDVI, brilho HSV e bordas Canny

**Pergunta científica:** quais dos três atributos originais contribuem para diferenciar árvores de gramíneas, e quais combinações permitem o melhor desempenho?

São avaliadas as **sete combinações não vazias** de:
- `mean_intensity` = brilho HSV;
- `mean_edges` = bordas Canny;
- `mean_ndvi` = NDVI.

A comparação é feita com **Random Forest (500 árvores), SVM RBF e regressão logística**. O **Dummy majoritário** é incluído como linha de referência — ele ignora características e, por isso, tem a mesma previsão nas sete combinações. Em todas as avaliações, os modelos compartilham **os mesmos blocos geográficos de teste**, a mesma deduplicação de objetos e a mesma faixa de exclusão de 25 metros já definidas no E2/E3. A padronização é ajustada só com o conjunto de treino. Não são utilizados novos dados.

**Como executar:** termine as células originais e execute **E2a**, que constrói `base_e2`; depois execute E4a e E4b. As células E3 não são obrigatórias para E4, mas, se executadas, permitem comparar o protocolo espacial completo com E3.

**Interpretação:** os experimentos medem classificação árvore/grama na área `Area1`, não aferem altura real nem interferência em cabos. Rodadas repetidas podem compartilhar dados; seus desvios-padrão são descritivos. Nenhum número novo de desempenho é afirmado antes de executar o Colab.


In [ ]:
# E4a — ablação das 7 combinações, com partições ESPACIAIS idênticas a E2/E3
# Requer base_e2/campos_e2 da célula E2a. Não muda modelos/saídas originais.
import itertools
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score

if "base_e2" not in globals() or "campos_e2" not in globals():
    raise RuntimeError("Execute E2a antes de E4: base_e2/campos_e2 ausentes.")
features_e4 = ["mean_intensity", "mean_edges", "mean_ndvi"]
nomes_features_e4 = {
    "mean_intensity": "Brilho HSV", "mean_edges": "Bordas Canny",
    "mean_ndvi": "NDVI"
}
assert all(f in base_e2.columns for f in features_e4)
assert base_e2.crs is not None and base_e2.crs.is_projected, "CRS deve ser métrico."
assert len(base_e2) == base_e2["segmento_id"].nunique(), "Segmentos duplicados em E2a."
assert set(base_e2["classe"].unique()) == {"arvore", "grama"}

X_total_e4 = base_e2[features_e4].to_numpy(dtype=float)
y_e4 = base_e2["classe"].to_numpy()
assert np.isfinite(X_total_e4).all(), "Atributos não finitos."
combinacoes_e4 = [
    tuple(combo)
    for k in range(1, len(features_e4) + 1)
    for combo in itertools.combinations(features_e4, k)
]
assert len(combinacoes_e4) == 7
centros_e4 = base_e2.geometry.centroid
xmin_e4, ymin_e4, _, _ = base_e2.total_bounds
linhas_e4, excluidos_e4 = [], []
particoes_validas_e4 = []

for tamanho_e4 in [250, 500]:
    col_e4 = np.floor((centros_e4.x.to_numpy() - xmin_e4) / tamanho_e4).astype(int)
    lin_e4 = np.floor((centros_e4.y.to_numpy() - ymin_e4) / tamanho_e4).astype(int)
    grupos_e4 = np.array([f"{c}_{l}" for c, l in zip(col_e4, lin_e4)])
    blocos_e4 = len(np.unique(grupos_e4))
    if blocos_e4 < 5:
        excluidos_e4.append((tamanho_e4, 0, "menos de cinco blocos"))
        continue

    partidor_e4 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e4, (tr_e4, te_e4) in enumerate(
        partidor_e4.split(X_total_e4, y_e4, groups=grupos_e4), start=1
    ):
        assert not (set(grupos_e4[tr_e4]) & set(grupos_e4[te_e4]))
        assert not (
            set(base_e2.iloc[tr_e4]["segmento_id"]) &
            set(base_e2.iloc[te_e4]["segmento_id"])
        ), "Vazamento de segmentos entre treino e teste."
        buffer_e4 = base_e2.geometry.iloc[te_e4].buffer(25)
        geometria_teste_e4 = (buffer_e4.union_all() if hasattr(buffer_e4, "union_all")
                              else buffer_e4.unary_union)
        afastados_e4 = ~base_e2.geometry.iloc[tr_e4].intersects(
            geometria_teste_e4
        ).to_numpy()
        tr_seguro_e4 = tr_e4[afastados_e4]
        if len(tr_seguro_e4) < 15 or len(te_e4) < 5:
            excluidos_e4.append((tamanho_e4, rodada_e4, "amostras insuficientes"))
            continue
        if (len(np.unique(y_e4[tr_seguro_e4])) < 2
                or len(np.unique(y_e4[te_e4])) < 2):
            excluidos_e4.append((tamanho_e4, rodada_e4, "classe ausente"))
            continue

        particoes_validas_e4.append((tamanho_e4, rodada_e4))
        # Dummy: apenas UMA predição por partição, pois não utiliza atributos.
        referencia_e4 = DummyClassifier(strategy="most_frequent")
        referencia_e4.fit(X_total_e4[tr_seguro_e4], y_e4[tr_seguro_e4])
        previsao_dummy_e4 = referencia_e4.predict(X_total_e4[te_e4])

        for combo_e4 in combinacoes_e4:
            idx_features_e4 = [features_e4.index(f) for f in combo_e4]
            X_combo_e4 = X_total_e4[:, idx_features_e4]
            etiqueta_e4 = " + ".join(nomes_features_e4[f] for f in combo_e4)
            modelos_e4 = {
                "Random Forest": RandomForestClassifier(
                    n_estimators=500, random_state=42, n_jobs=-1
                ),
                "SVM (RBF)": make_pipeline(
                    StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")
                ),
                "Regressão logística": make_pipeline(
                    StandardScaler(),
                    LogisticRegression(max_iter=2000, random_state=42)
                ),
            }
            for nome_e4, modelo_e4 in modelos_e4.items():
                modelo_e4.fit(X_combo_e4[tr_seguro_e4], y_e4[tr_seguro_e4])
                previsao_e4 = modelo_e4.predict(X_combo_e4[te_e4])
                linhas_e4.append({
                    "bloco_m": tamanho_e4, "rodada": rodada_e4, "modelo": nome_e4,
                    "atributos": etiqueta_e4, "quantidade_atributos": len(combo_e4),
                    "variaveis": "|".join(combo_e4),
                    "n_treino": len(tr_seguro_e4), "n_teste": len(te_e4),
                    "acuracia": accuracy_score(y_e4[te_e4], previsao_e4),
                    "acuracia_balanceada": balanced_accuracy_score(
                        y_e4[te_e4], previsao_e4
                    ),
                    "f1_macro": f1_score(
                        y_e4[te_e4], previsao_e4, average="macro", zero_division=0
                    ),
                    "f1_arvore": f1_score(
                        y_e4[te_e4], previsao_e4, pos_label="arvore", zero_division=0
                    ),
                    "recall_arvore": recall_score(
                        y_e4[te_e4], previsao_e4, pos_label="arvore", zero_division=0
                    ),
                })
            # A referência Dummy é a mesma em todas as ablações desta partição.
            linhas_e4.append({
                "bloco_m": tamanho_e4, "rodada": rodada_e4, "modelo": "Dummy (maioria)",
                "atributos": etiqueta_e4, "quantidade_atributos": len(combo_e4),
                "variaveis": "|".join(combo_e4),
                "n_treino": len(tr_seguro_e4), "n_teste": len(te_e4),
                "acuracia": accuracy_score(y_e4[te_e4], previsao_dummy_e4),
                "acuracia_balanceada": balanced_accuracy_score(
                    y_e4[te_e4], previsao_dummy_e4
                ),
                "f1_macro": f1_score(
                    y_e4[te_e4], previsao_dummy_e4, average="macro", zero_division=0
                ),
                "f1_arvore": f1_score(
                    y_e4[te_e4], previsao_dummy_e4, pos_label="arvore", zero_division=0
                ),
                "recall_arvore": recall_score(
                    y_e4[te_e4], previsao_dummy_e4, pos_label="arvore", zero_division=0
                ),
            })

tabela_e4 = pd.DataFrame(linhas_e4)
print("=== E4 | Ablação dos atributos com Area1 ===")
print("Combinações:", len(combinacoes_e4), "| Modelos: 4 (3 supervisionados + Dummy)")
print("Partições espaciais válidas:", len(particoes_validas_e4))
print("Partições descartadas:", len(excluidos_e4))
for tam_e4, num_e4, causa_e4 in excluidos_e4:
    print(f"  {tam_e4} m / rodada {num_e4}: {causa_e4}")
if tabela_e4.empty:
    print("Sem partições válidas — nenhum desempenho novo foi produzido.")
else:
    totais_e4 = tabela_e4.groupby(["bloco_m", "rodada", "modelo"]).size()
    assert (totais_e4 == 7).all(), "Alguma ablação está ausente numa partição."
    print("\nResumo descritivo — F1-macro por escala / modelo / atributos:")
    print(
        tabela_e4.groupby(["bloco_m", "modelo", "atributos"])[
            ["f1_macro", "acuracia_balanceada", "recall_arvore"]
        ].mean().round(3).to_string()
    )


In [ ]:
# E4b — comparação PAREADA de cada subconjunto com as três características originais
# O mesmo conjunto de teste é utilizado em todas as linhas de cada partição.
import matplotlib.pyplot as plt

if tabela_e4.empty:
    print("E4 sem rodadas válidas; não há comparações nem gráficos.")
else:
    full_key_e4 = "|".join(features_e4)
    colunas_chave_e4 = ["bloco_m", "rodada", "modelo"]
    full_e4 = tabela_e4.loc[
        tabela_e4["variaveis"] == full_key_e4,
        colunas_chave_e4 + ["f1_macro", "acuracia_balanceada"]
    ].rename(columns={
        "f1_macro": "f1_full", "acuracia_balanceada": "balanceada_full"
    })
    comparacoes_e4 = tabela_e4.merge(
        full_e4, on=colunas_chave_e4, how="left", validate="many_to_one"
    )
    assert comparacoes_e4["f1_full"].notna().all()
    comparacoes_e4["delta_f1_vs_full"] = (
        comparacoes_e4["f1_macro"] - comparacoes_e4["f1_full"]
    )
    print("=== E4 | Diferença pareada de F1-macro: subconjunto MENOS os 3 atributos ===")
    supervis_e4 = comparacoes_e4.loc[
        comparacoes_e4["modelo"] != "Dummy (maioria)"
    ]
    deltas_e4 = supervis_e4.groupby(
        ["bloco_m", "modelo", "atributos"]
    )["delta_f1_vs_full"].agg(["count", "mean", "std"]).round(3)
    print(deltas_e4.to_string())

    # Análise de retirada individual: o efeito é observacional e depende da amostra.
    print("\n=== E4 | Retirada individual de atributos (F1 full MENOS F1 sem atributo) ===")
    for removida_e4 in features_e4:
        faltando_e4 = "|".join(f for f in features_e4 if f != removida_e4)
        analise_e4 = supervis_e4.loc[
            supervis_e4["variaveis"] == faltando_e4
        ].copy()
        analise_e4["perda_ao_remover"] = -analise_e4["delta_f1_vs_full"]
        print("\nExclusão:", nomes_features_e4[removida_e4])
        print(analise_e4.groupby(["bloco_m", "modelo"])["perda_ao_remover"]
              .agg(["count", "mean", "std"]).round(3).to_string())

    for tamanho_graf_e4 in [250, 500]:
        subconjunto_graf_e4 = supervis_e4.loc[
            supervis_e4["bloco_m"] == tamanho_graf_e4
        ]
        if subconjunto_graf_e4.empty:
            continue
        colunas_e4 = [
            "Brilho HSV", "Bordas Canny", "NDVI",
            "Brilho HSV + Bordas Canny", "Brilho HSV + NDVI",
            "Bordas Canny + NDVI", "Brilho HSV + Bordas Canny + NDVI"
        ]
        medias_e4 = subconjunto_graf_e4.pivot_table(
            index="atributos", columns="modelo", values="f1_macro", aggfunc="mean"
        ).reindex(colunas_e4)
        fig, ax = plt.subplots(figsize=(12, 5))
        medias_e4.plot(kind="bar", ax=ax)
        ax.set_title(f"E4 — F1-macro médio por atributos e modelo ({tamanho_graf_e4} m)")
        ax.set_xlabel("Combinação de características")
        ax.set_ylabel("F1-macro (média descritiva)")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=35)
        fig.tight_layout()
        plt.show()

    # Checagem contra os métodos do E3, se E3 também foi executado na sessão.
    if ("tabela_e3" in globals() and isinstance(tabela_e3, pd.DataFrame)
            and not tabela_e3.empty):
        full_e4_check = tabela_e4.loc[
            tabela_e4["variaveis"] == full_key_e4,
            ["bloco_m", "rodada", "modelo", "n_treino", "n_teste", "f1_macro"]
        ]
        comparavel_e3 = tabela_e3[
            ["bloco_m", "rodada", "modelo", "n_treino", "n_teste", "f1_macro"]
        ]
        teste_e3e4 = full_e4_check.merge(
            comparavel_e3,
            on=["bloco_m", "rodada", "modelo"],
            suffixes=("_e4", "_e3"),
            how="outer", indicator=True, validate="one_to_one"
        )
        assert (teste_e3e4["_merge"] == "both").all(), (
            "As partições/modelos de E3 e E4 não são idênticos."
        )
        for metrica_e4 in ["n_treino", "n_teste", "f1_macro"]:
            assert np.allclose(
                teste_e3e4[f"{metrica_e4}_e3"],
                teste_e3e4[f"{metrica_e4}_e4"]
            ), f"E3 difere de E4 com os 3 atributos: {metrica_e4}"
        print("\nChecagem E3 × E4: métricas idênticas com os três atributos.")

    print("\nATENÇÃO: ganhos ou perdas aqui são descritivos na mesma Area1.")
    print("Não inferir superioridade geral nem importância causal sem validação externa.")


## E5 — sensibilidade do Quickshift e do K-Means (dados da Area1)

**Objetivo:** medir como mudanças de parâmetros afetam a segmentação e o agrupamento, sem modificar as 116 células históricas nem exigir novos dados.

São **dois ensaios distintos**, pois alterar o Quickshift muda a identidade e a geometria dos segmentos; não seria válido reutilizar, sem refazer a associação dos rótulos e a extração de atributos, as mesmas linhas de treino/teste de E2–E4.

- **E5a — Quickshift:** aplica 5 configurações em **quatro janelas fixas de 256 × 256 pixels**, escolhidas deterministicamente perto das áreas de referência para árvores e gramíneas já existentes. Mede tempo de execução, número de segmentos, mediana da área de segmentos e fração de segmentos muito pequenos. A configuração original é `kernel_size=3, max_dist=5, ratio=0.04`. **Os resultados das janelas não são equivalentes aos do raster inteiro.**
- **E5b — K-Means:** sobre **uma amostra fixa de até 1.500 polígonos** da segmentação original (priorizando amostras com rótulo e incluindo amostra do restante), extrai das mesmas bandas RGB/NIR as oito características originais (médias e desvios-padrão). Testa **K = 4, 6, 8** com `random_state=42, n_init=10`, sem padronização, como no código original. Mede inércia, silhouette (amostral), tempo, estabilidade de grupos por **Adjusted Rand Index (ARI)** e distribuição de classes rotuladas nos grupos.
- **Memória:** exibe somente o tamanho aproximado dos principais arrays de entrada e saída; **não** equivale ao pico completo de RAM da execução. Os resultados dependem de executar no Colab.
- **Rótulos e clusters:** IDs de clusters do K-Means são arbitrários (não assumir que `cluster 3` permanece sendo vegetação). A distribuição de classes por cluster é exploratória, pois as referências podem intersectar múltiplos segmentos e não constituem validação independente.
- **Sem novas amostras nem medidas:** os rótulos já existentes são usados apenas para selecionar as janelas e contextualizar a análise. Não é medida de precisão de detecção ou risco elétrico.

**Execução:** depois do pipeline original na V2, execute E5a e E5b. E5b utiliza `clip_poly`, `gdf_full` e `path_img_mult` produzidos pelo notebook. Não é necessário executar E1–E4 para esses dois ensaios. Se a memória for limitada, execute **E5a antes** de E5b e evite manter cópias extras dos rasters.


In [ ]:
# E5a — sensibilidade do QUICKSHIFT em janelas fixas da imagem original
# NÃO substitui segments_quick2 nem altera im/src_rgb ou suas saídas históricas.
import time
import numpy as np
import pandas as pd
from rasterio.transform import rowcol
from skimage.segmentation import quickshift

assert "src_rgb" in globals() and "im" in globals(), "Execute a leitura RGB original."
assert "gdf_arvore" in globals() and "gdf_grama" in globals(), (
    "Execute a leitura das camadas Arvores.gpkg e Grama.gpkg."
)
assert im.ndim == 3 and im.shape[2] >= 3
assert src_rgb.crs is not None
assert (src_rgb.height, src_rgb.width) == im.shape[:2], "Raster e imagem não alinhados."
if min(src_rgb.height, src_rgb.width) < 256:
    raise ValueError("A imagem original deve permitir janelas de 256 x 256 pixels.")

configuracoes_e5_q = [
    (2, 5, 0.04), (3, 3, 0.04), (3, 5, 0.04),
    (3, 8, 0.04), (4, 5, 0.04)
]
lado_e5_q = 256
janelas_e5_q = []
vistos_e5_q = set()

# Dois centros determinísticos por classe, quando houver; base já existente.
for classe_e5_q, camada_e5_q in [("arvore", gdf_arvore), ("grama", gdf_grama)]:
    candidatos_e5_q = camada_e5_q.to_crs(src_rgb.crs)
    candidatos_e5_q = candidatos_e5_q.loc[
        candidatos_e5_q.geometry.notna()
        & ~candidatos_e5_q.geometry.is_empty
    ].copy()
    centroides_e5_q = candidatos_e5_q.geometry.centroid
    locais_e5_q = []
    for centro_e5_q in centroides_e5_q:
        rr_e5_q, cc_e5_q = rowcol(src_rgb.transform, centro_e5_q.x, centro_e5_q.y)
        if 0 <= rr_e5_q < src_rgb.height and 0 <= cc_e5_q < src_rgb.width:
            r0_e5_q = int(np.clip(rr_e5_q - lado_e5_q // 2, 0,
                                  src_rgb.height - lado_e5_q))
            c0_e5_q = int(np.clip(cc_e5_q - lado_e5_q // 2, 0,
                                  src_rgb.width - lado_e5_q))
            if (r0_e5_q, c0_e5_q) not in vistos_e5_q:
                locais_e5_q.append((r0_e5_q, c0_e5_q))
    # Permite manter janelas em diferentes regiões da mesma classe.
    if locais_e5_q:
        escolhas_e5_q = [0, len(locais_e5_q) - 1]
        for ix_e5_q in escolhas_e5_q:
            if len(janelas_e5_q) >= 4:
                break
            coords_e5_q = locais_e5_q[ix_e5_q]
            if coords_e5_q not in vistos_e5_q:
                janelas_e5_q.append((classe_e5_q, *coords_e5_q))
                vistos_e5_q.add(coords_e5_q)

if not janelas_e5_q:
    raise ValueError("Nenhuma janela válida encontrada junto aos rótulos existentes.")
if len(janelas_e5_q) < 4:
    print("Aviso: somente", len(janelas_e5_q),
          "janelas únicas encontradas; as configurações ainda serão comparadas nas mesmas janelas.")

linhas_e5_quick = []
for ind_e5_q, (classe_e5_q, r0_e5_q, c0_e5_q) in enumerate(janelas_e5_q, 1):
    recorte_e5_q = np.ascontiguousarray(
        im[r0_e5_q:r0_e5_q + lado_e5_q, c0_e5_q:c0_e5_q + lado_e5_q, :3]
    )
    for kernel_e5_q, dist_e5_q, ratio_e5_q in configuracoes_e5_q:
        inicio_e5_q = time.perf_counter()
        seg_e5_q = quickshift(
            recorte_e5_q, kernel_size=kernel_e5_q,
            max_dist=dist_e5_q, ratio=ratio_e5_q
        )
        segundos_e5_q = time.perf_counter() - inicio_e5_q
        _, contagens_e5_q = np.unique(seg_e5_q, return_counts=True)
        linhas_e5_quick.append({
            "janela": ind_e5_q, "classe_referencia_centro": classe_e5_q,
            "row0": r0_e5_q, "col0": c0_e5_q,
            "kernel_size": kernel_e5_q, "max_dist": dist_e5_q,
            "ratio": ratio_e5_q, "config_original":
                kernel_e5_q == 3 and dist_e5_q == 5 and ratio_e5_q == 0.04,
            "n_segmentos": int(len(contagens_e5_q)),
            "pixels_medianos_segmento": float(np.median(contagens_e5_q)),
            "fracao_segmentos_menos_10px": float(
                np.mean(contagens_e5_q < 10)
            ),
            "tempo_s": segundos_e5_q,
            "memoria_arrays_mb": (recorte_e5_q.nbytes + seg_e5_q.nbytes) / 2**20,
        })
    print(f"Janela E5a {ind_e5_q}/{len(janelas_e5_q)} concluída.")
tabela_e5_quick = pd.DataFrame(linhas_e5_quick)
print("\n=== E5a — Quickshift, mesmas janelas (256x256) ===")
print(tabela_e5_quick.round(4).to_string(index=False))
print("\nResumo por parâmetros (média entre janelas, não teste independente):")
print(tabela_e5_quick.groupby(
    ["kernel_size", "max_dist", "ratio", "config_original"]
)[["n_segmentos", "pixels_medianos_segmento",
   "fracao_segmentos_menos_10px", "tempo_s"]].agg(["mean", "std"]).round(4))
print("\nNOTA: tempo/memória em janelas pequenas NÃO extrapolam linearmente ao raster inteiro.")


In [ ]:
# E5b — K-Means em uma amostra FIXA de segmentos originais, apenas Area1
# Usa polígonos e bandas multiespectrais existentes; nenhuma segmentação/reclassificação é sobrescrita.
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

assert "clip_poly" in globals() and "gdf_full" in globals(), (
    "Execute as células de clip_poly e gdf_full no pipeline original."
)
assert "path_img_mult" in globals(), "Falta a imagem multiespectral original."
assert clip_poly.crs is not None and gdf_full.crs is not None
if len(clip_poly) < 24:
    raise ValueError("São necessários 24 segmentos originais para K=8.")
# Amostra fixa com segmentos anotados por interseção espacial e parte não anotada.
geometrias_e5_k = clip_poly.loc[
    clip_poly.geometry.notna() & ~clip_poly.geometry.is_empty
].copy()
geometrias_e5_k = geometrias_e5_k.reset_index(drop=True)
referencias_e5_k = gdf_full[["Classe", "geometry"]].to_crs(geometrias_e5_k.crs)
encontro_e5_k = geometrias_e5_k[["geometry"]].sjoin(
    referencias_e5_k, predicate="intersects", how="inner"
)
# Segmentos com rótulos conflitantes ficam fora da análise de classe; NÃO são excluídos do K-Means.
rotulos_e5_k = encontro_e5_k.groupby(level=0)["Classe"].agg(
    lambda x: x.iloc[0] if x.nunique() == 1 else "conflitante"
)
alvos_e5_k = np.array(sorted(rotulos_e5_k.index), dtype=int)
rng_e5_k = np.random.default_rng(42)
max_amostra_e5_k = min(1500, len(geometrias_e5_k))
if len(alvos_e5_k) > max_amostra_e5_k // 2:
    alvos_e5_k = np.sort(rng_e5_k.choice(
        alvos_e5_k, size=max_amostra_e5_k // 2, replace=False
    ))
restantes_e5_k = np.setdiff1d(
    np.arange(len(geometrias_e5_k)), alvos_e5_k, assume_unique=False
)
qtd_outros_e5_k = max_amostra_e5_k - len(alvos_e5_k)
outros_e5_k = rng_e5_k.choice(
    restantes_e5_k, size=qtd_outros_e5_k, replace=False
)
indices_e5_k = np.sort(np.concatenate([alvos_e5_k, outros_e5_k]))
amostra_pol_e5_k = geometrias_e5_k.iloc[indices_e5_k].copy()
lista_e5_k = []
nomes_e5_k = ["R_Mean", "G_Mean", "B_Mean", "N_Mean",
               "R_Std", "G_Std", "B_Std", "N_Std"]

with rasterio.open(path_img_mult) as raster_e5_k:
    if raster_e5_k.count < 4:
        raise ValueError("K-Means E5 requer quatro bandas RGB/NIR no raster original.")
    if amostra_pol_e5_k.crs != raster_e5_k.crs:
        amostra_pol_e5_k = amostra_pol_e5_k.to_crs(raster_e5_k.crs)
    for id_local_e5_k, obj_e5_k in amostra_pol_e5_k.iterrows():
        try:
            imagem_e5_k, _ = mask(
                raster_e5_k, [obj_e5_k.geometry], crop=True, nodata=0
            )
        except (ValueError, rasterio.errors.WindowError):
            continue  # Pode estar fora do raster multiespectral devido à diferença de origem.
        medias_std_e5_k = []
        for banda_e5_k in imagem_e5_k[:4]:
            valores_e5_k = banda_e5_k[banda_e5_k != 0]
            medias_std_e5_k.append(float(np.mean(valores_e5_k)) if valores_e5_k.size else np.nan)
        for banda_e5_k in imagem_e5_k[:4]:
            valores_e5_k = banda_e5_k[banda_e5_k != 0]
            medias_std_e5_k.append(float(np.std(valores_e5_k)) if valores_e5_k.size else np.nan)
        lista_e5_k.append((id_local_e5_k, *medias_std_e5_k))
amostras_e5_k = pd.DataFrame(
    lista_e5_k, columns=["segmento", *nomes_e5_k]
).set_index("segmento")
amostras_e5_k = amostras_e5_k.replace([np.inf, -np.inf], np.nan).dropna()
if len(amostras_e5_k) < 24:
    raise ValueError("Após extrair bandas, restaram menos de 24 segmentos utilizáveis.")
X_e5_k = amostras_e5_k[nomes_e5_k].to_numpy(dtype=float)
metricas_e5_k = []
predicoes_e5_k = {}
for k_e5_k in [4, 6, 8]:
    inicio_e5_k = time.perf_counter()
    modelo_e5_k = KMeans(n_clusters=k_e5_k, random_state=42, n_init=10)
    grupos_e5_k = modelo_e5_k.fit_predict(X_e5_k)
    tempo_e5_k = time.perf_counter() - inicio_e5_k
    predicoes_e5_k[k_e5_k] = grupos_e5_k
    # Silhouette amostral com mesmos itens; não é medida de identificação correta das árvores.
    if 1 < len(np.unique(grupos_e5_k)) < len(X_e5_k):
        sil_e5_k = silhouette_score(
            X_e5_k, grupos_e5_k, sample_size=min(len(X_e5_k), 500),
            random_state=42
        )
    else:
        sil_e5_k = np.nan  # Agrupamento degenerado: silhouette indefinida
    metricas_e5_k.append({
        "k": k_e5_k, "segmentos_validos": len(X_e5_k),
        "inercia": modelo_e5_k.inertia_,
        "silhouette_amostral": sil_e5_k,
        "tempo_s": tempo_e5_k,
        "memoria_matriz_mb": X_e5_k.nbytes / 2**20,
        "ari_vs_k6": adjusted_rand_score(
            predicoes_e5_k[6], grupos_e5_k
        ) if 6 in predicoes_e5_k else np.nan
    })
# K=4 precede K=6; preencher ARI de K=4 em relação ao K=6 após os ajustes.
for resultado_e5_k in metricas_e5_k:
    resultado_e5_k["ari_vs_k6"] = adjusted_rand_score(
        predicoes_e5_k[6], predicoes_e5_k[resultado_e5_k["k"]]
    )
tabela_e5_kmeans = pd.DataFrame(metricas_e5_k)
print("\n=== E5b — sensibilidade K-Means na mesma amostra ===")
print(tabela_e5_kmeans.round(4).to_string(index=False))
for k_e5_k, pred_e5_k in predicoes_e5_k.items():
    print(f"\nK={k_e5_k} — quantidade de segmentos por cluster (IDs sem significado fixo):")
    print(pd.Series(pred_e5_k).value_counts().sort_index().to_string())
    lista_rot_e5_k = rotulos_e5_k.reindex(amostras_e5_k.index).fillna("sem_rotulo")
    tab_e5_k = pd.crosstab(
        pd.Series(pred_e5_k, name="cluster"), lista_rot_e5_k.reset_index(drop=True),
        dropna=False
    )
    print("Cruzamento exploratório com referências já existentes:")
    print(tab_e5_k.to_string())
print("\nATENÇÃO: cluster 3 em K=4, 6 ou 8 NÃO designa a mesma categoria.")
print("Silhouette e ARI medem estrutura/estabilidade do agrupamento,")
print("NÃO precisão da detecção de árvores e NÃO validam a seleção da classe 3.")


## E6 — proximidade a postes: com e sem filtro de classe

**Objetivo:** auditar o resultado histórico da etapa espacial usando **apenas os dados já existentes da Area1**. O fluxo original inclui objetos com `Power_tree_size_C3 > 7` mesmo quando o classificador os marcou como gramínea. Este experimento compara a seleção histórica com a seleção que exige simultaneamente `predicted_classe == 'arvore'` e o mesmo limiar C3. A categoria "grama" é exibida separadamente para ajudar a interpretar a diferença.

**Três cenários:** (1) todos os candidatos C3 > 7, (2) somente os candidatos classificados como árvore, (3) somente os classificados como gramínea. Os cenários utilizam **exatamente os mesmos polígonos, critérios C3 e postes**; somente a condição de classe muda.

**Três distâncias:** 1, 3 e 5 metros. Para evitar inflar as contagens, a E6 usa `sjoin_nearest` em **ambos os sentidos**, uma vez por cenário e com `max_distance=5`; depois filtra a menor distância de cada objeto para cada limiar. Assim, contabiliza tanto **segmentos distintos** quanto **postes distintos** próximos aos segmentos. Não soma as linhas de associações com empates: um poste ou segmento pode aparecer em mais de uma linha, mas é contado apenas uma vez.

**Condições e limitações:** as distâncias são **planimétricas entre polígonos e pontos de postes**, não distâncias aos cabos ou medidas de segurança. `C3 > 7` é apenas uma estimativa empírica de porte, **não altura observada**. O classificador é o **RF histórico**; este teste não demonstra melhoria de acurácia ou risco elétrico real. A E6 não modifica `filter_tree`, `tree_polys`, `Postes`, os 116 blocos originais nem os arquivos de entrada.

**Execução:** execute primeiro as células originais até gerar `tree_polys`, `filter_tree` e `Postes` já reprojetados e recortados para a AOI; depois execute E6a e E6b. **E1–E5 não são pré-requisitos para E6.** Não há resultados novos até executar com os dados reais no Colab.


In [ ]:
# E6a — contagens únicas de segmentos e postes em 1, 3 e 5 m
# Pré-requisitos: execução das células originais até Postes = Postes.clip(AOI).
import numpy as np
import pandas as pd
import geopandas as gpd

for nome_e6 in ["tree_polys", "filter_tree", "Postes"]:
    if nome_e6 not in globals():
        raise RuntimeError(f"Execute as células originais antes da E6: falta {nome_e6}.")
if tree_polys.crs is None or Postes.crs is None:
    raise ValueError("CRS ausente nas geometrias.")
if tree_polys.index.has_duplicates:
    raise ValueError("IDs de segmentos repetidos em tree_polys. Não deduplicar sem auditoria.")
colunas_e6 = ["Power_tree_size_C3", "predicted_classe"]
if any(nome_e6 not in tree_polys.columns for nome_e6 in colunas_e6):
    raise ValueError("Faltam o porte C3 ou as previsões do classificador RF original.")

# CRS projetado em metros, correspondente ao raster original da Area1.
objetos_e6 = tree_polys[["Power_tree_size_C3", "predicted_classe", "geometry"]].copy()
objetos_e6 = objetos_e6.to_crs(epsg=31982)
objetos_e6["segmento_id_e6"] = objetos_e6.index.map(str)
objetos_e6["classe_e6"] = (
    objetos_e6["predicted_classe"].astype(str).str.strip().str.lower()
)
objetos_e6["porte_c3_e6"] = pd.to_numeric(
    objetos_e6["Power_tree_size_C3"], errors="coerce"
)
objetos_e6 = objetos_e6.loc[
    objetos_e6.geometry.notna() & (~objetos_e6.geometry.is_empty) &
    np.isfinite(objetos_e6["porte_c3_e6"])
].copy()
if not objetos_e6.geometry.is_valid.all():
    raise ValueError("Geometrias de candidatos inválidas; reveja os polígonos.")
candidatos_e6 = objetos_e6.loc[objetos_e6["porte_c3_e6"] > 7].copy()
candidatos_e6 = candidatos_e6.reset_index(drop=True)
if len(candidatos_e6) != len(filter_tree):
    print("AVISO: a contagem E6 difere de filter_tree (geometrias vazias/inválidas ou C3 não finito).")

postes_brutos_e6 = Postes[["geometry"]].copy().to_crs(candidatos_e6.crs)
postes_brutos_e6 = postes_brutos_e6.loc[
    postes_brutos_e6.geometry.notna() & (~postes_brutos_e6.geometry.is_empty)
].copy()
if not postes_brutos_e6.geometry.is_valid.all():
    raise ValueError("Geometrias inválidas na camada de postes.")
if not postes_brutos_e6.geometry.geom_type.isin(["Point"]).all():
    raise ValueError("A E6 requer postes representados por pontos.")
# Registros com geometrias idênticas são uma única localização de poste.
postes_brutos_e6["chave_geom_e6"] = postes_brutos_e6.geometry.to_wkb(hex=True)
postes_e6 = postes_brutos_e6.drop_duplicates("chave_geom_e6").copy()
postes_e6 = postes_e6[["geometry"]].reset_index(drop=True)
postes_e6["poste_id_e6"] = np.arange(len(postes_e6), dtype=int)
print("=== E6 | Conferência dos dados ===")
print("Candidatos com C3 > 7:", len(candidatos_e6))
print("Classes previstas entre candidatos:",
      candidatos_e6["classe_e6"].value_counts().to_dict())
print("Postes válidos (antes de retirar coordenadas idênticas):", len(postes_brutos_e6))
print("Postes em localizações distintas:", len(postes_e6))
if not candidatos_e6.empty and not postes_e6.empty:
    assert candidatos_e6["segmento_id_e6"].is_unique
    assert postes_e6["poste_id_e6"].is_unique

cenarios_e6 = {
    "C3 > 7 (histórico)": candidatos_e6,
    "C3 > 7 e árvore": candidatos_e6.loc[candidatos_e6["classe_e6"] == "arvore"],
    "C3 > 7 e gramínea": candidatos_e6.loc[candidatos_e6["classe_e6"] == "grama"],
}
raios_e6 = [1, 3, 5]  # metros, em EPSG:31982
linhas_e6 = []
# Duas direções são necessárias: nearest(polygon -> pole) conta polígonos,
# nearest(pole -> polygon) conta postes. Ambos usam a mesma distância exata.
for cenario_e6, poligonos_e6 in cenarios_e6.items():
    total_candidatos_e6 = len(poligonos_e6)
    if total_candidatos_e6 and len(postes_e6):
        sjoin_seg_e6 = gpd.sjoin_nearest(
            poligonos_e6[["segmento_id_e6", "geometry"]],
            postes_e6[["poste_id_e6", "geometry"]],
            how="inner", max_distance=5, distance_col="dist_m_e6"
        )
        sjoin_post_e6 = gpd.sjoin_nearest(
            postes_e6[["poste_id_e6", "geometry"]],
            poligonos_e6[["segmento_id_e6", "geometry"]],
            how="inner", max_distance=5, distance_col="dist_m_e6"
        )
    else:
        sjoin_seg_e6 = pd.DataFrame(columns=["segmento_id_e6", "dist_m_e6"])
        sjoin_post_e6 = pd.DataFrame(columns=["poste_id_e6", "dist_m_e6"])
    for raio_e6 in raios_e6:
        proximos_seg_e6 = sjoin_seg_e6.loc[
            sjoin_seg_e6["dist_m_e6"] <= raio_e6 + 1e-9
        ]
        proximos_post_e6 = sjoin_post_e6.loc[
            sjoin_post_e6["dist_m_e6"] <= raio_e6 + 1e-9
        ]
        segmentos_proximos_e6 = proximos_seg_e6["segmento_id_e6"].nunique()
        postes_proximos_e6 = proximos_post_e6["poste_id_e6"].nunique()
        linhas_e6.append({
            "cenario": cenario_e6, "distancia_m": raio_e6,
            "segmentos_candidatos": total_candidatos_e6,
            "segmentos_proximos": int(segmentos_proximos_e6),
            "fracao_segmentos_pct": round(
                100 * segmentos_proximos_e6 / total_candidatos_e6, 2
            ) if total_candidatos_e6 else np.nan,
            "postes_total": len(postes_e6),
            "postes_proximos": int(postes_proximos_e6),
            "fracao_postes_pct": round(
                100 * postes_proximos_e6 / len(postes_e6), 2
            ) if len(postes_e6) else np.nan,
        })

tabela_e6 = pd.DataFrame(linhas_e6)
print("\n=== E6 | Contagens únicas por cenário e distância ===")
print(tabela_e6.to_string(index=False))


In [ ]:
# E6b — comparação entre método histórico e método filtrado por classe
# Relatório descritivo, sem inferir validação de altura ou risco de interferência.
import matplotlib.pyplot as plt

if "tabela_e6" not in globals() or tabela_e6.empty:
    raise RuntimeError("Execute E6a antes de E6b.")

historico_e6 = tabela_e6.loc[
    tabela_e6["cenario"] == "C3 > 7 (histórico)"
].set_index("distancia_m")
arvore_e6 = tabela_e6.loc[
    tabela_e6["cenario"] == "C3 > 7 e árvore"
].set_index("distancia_m")
grama_e6 = tabela_e6.loc[
    tabela_e6["cenario"] == "C3 > 7 e gramínea"
].set_index("distancia_m")
comparativo_e6 = pd.DataFrame({
    "distancia_m": historico_e6.index,
    "segmentos_historico": historico_e6["segmentos_proximos"].to_numpy(),
    "segmentos_arvore": arvore_e6["segmentos_proximos"].to_numpy(),
    "segmentos_grama": grama_e6["segmentos_proximos"].to_numpy(),
    "postes_historico": historico_e6["postes_proximos"].to_numpy(),
    "postes_arvore": arvore_e6["postes_proximos"].to_numpy(),
    "postes_grama": grama_e6["postes_proximos"].to_numpy(),
})
comparativo_e6["segmentos_excluidos_pelo_filtro"] = (
    comparativo_e6["segmentos_historico"] - comparativo_e6["segmentos_arvore"]
)
comparativo_e6["postes_deixam_de_ser_sinalizados"] = (
    comparativo_e6["postes_historico"] - comparativo_e6["postes_arvore"]
)
for coluna_e6 in [
    "segmentos_historico", "segmentos_arvore", "segmentos_grama",
    "postes_historico", "postes_arvore", "postes_grama"
]:
    assert (comparativo_e6[coluna_e6].diff().fillna(0) >= 0).all(), (
        f"Não monotonicidade na contagem {coluna_e6}; verificar as associações."
    )
assert (comparativo_e6["segmentos_excluidos_pelo_filtro"] >= 0).all()
assert (comparativo_e6["postes_deixam_de_ser_sinalizados"] >= 0).all()
assert (comparativo_e6["segmentos_historico"] >= comparativo_e6["segmentos_grama"]).all()
# Dentro dos candidatos históricos, as categorias são disjuntas.
# Se existirem outras classes inesperadas, a tabela ainda é válida, mas não somamos como árvore/grama.
print("=== E6 | Comparação do histórico e do filtro de classificação ===")
print(comparativo_e6.to_string(index=False))
print("\nDiferença em segmentos: candidatos próximos que deixam de ser selecionados")
print("quando se exige a previsão de árvore (não são erros comprovados de classificação).")
print("Diferença em postes: localizações próximas que deixam de ser sinalizadas.")
print("Um poste pode estar próximo de segmentos de ambas as classes; contagens de postes")
print("para árvore e gramínea não devem ser somadas para obter o total histórico.")

fig_e6, ax_e6 = plt.subplots(1, 2, figsize=(11, 4))
for nome_e6, eixo_e6 in [
    ("segmentos", ax_e6[0]), ("postes", ax_e6[1])
]:
    eixo_e6.plot(
        comparativo_e6["distancia_m"],
        comparativo_e6[f"{nome_e6}_historico"],
        marker="o", label="C3 > 7 (histórico)"
    )
    eixo_e6.plot(
        comparativo_e6["distancia_m"],
        comparativo_e6[f"{nome_e6}_arvore"],
        marker="s", label="C3 > 7 + árvore"
    )
    eixo_e6.set_xticks([1, 3, 5])
    eixo_e6.set_xlabel("Distância plana ao poste (m)")
    eixo_e6.set_ylabel(f"Número de {nome_e6} distintos")
    eixo_e6.grid(alpha=0.2)
    eixo_e6.legend()
fig_e6.suptitle("E6 — sensibilidade do filtro de classe na proximidade a postes")
fig_e6.tight_layout()
plt.show()
print("\nLIMITAÇÕES: a E6 mede proximidade geométrica a pontos de postes.")
print("Não há geometrias de cabos nem alturas reais que permitam estimar risco elétrico.")
